# Prepare the training data (CPU session)

In [1]:
import subprocess, glob
print(subprocess.run("python --version; free -g | head -2; df -h /kaggle/working | tail -1; nvidia-smi -L || echo 'no GPU (expected for this notebook)'",
                     shell=True, capture_output=True, text=True).stdout)
print("STEAD files found:", glob.glob("/kaggle/input/**/merged.*", recursive=True))

Python 3.13.18
               total        used        free      shared  buff/cache   available
Mem:              31           1          25           0           4          30
/dev/loop2       20G   88K   20G   1% /kaggle/working
no GPU (expected for this notebook)

STEAD files found: ['/kaggle/input/datasets/prabhanshu22/dat-repo-data/merged.csv', '/kaggle/input/datasets/prabhanshu22/dat-repo-data/merged.hdf5']


In [ ]:
# Clone the repository 
!pip install -q obspy torchinfo

In [ ]:
import glob, os, shutil, subprocess

REPO = "/kaggle/working/repo"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/yinjiuxun/WaveDecompNet-paper.git", REPO], check=True)
print("repo commit:", subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())

hit = glob.glob("/kaggle/input/**/step2b_prepare_STEAD_waveforms.py", recursive=True)
assert hit, "Add the wdn-code dataset (the 8 files from wdn_code.zip) to this notebook via Add Input."
src_dir = os.path.dirname(hit[0])
for f in ["config.py", "wdn_common.py", "step2a_download_poha.py", "step2b_prepare_STEAD_waveforms.py",
          "step2c_prepare_continous_STEAD_waveforms.py", "step2d_combine_all_data.py",
          "step3_training_1D.py", "PATCH_NOTES.md"]:
    shutil.copy(os.path.join(src_dir, f), os.path.join(REPO, f))
p = os.path.join(REPO, "torch_tools.py")
s = open(p).read()
open(p, "w").write(s.replace("np.Inf", "np.inf"))
print("patched files copied; np.Inf -> np.inf done")

Cloning into '/kaggle/working/repo'...


repo commit: afe2536cf29ead8b4d8d5c50ddf85cc1dd90ddd1
patched files copied; np.Inf -> np.inf done


In [ ]:
# Inspect the STEAD files

%cd /kaggle/working/repo
import config as C, h5py, pandas as pd
print("hdf5:", C.STEAD_HDF5)
print("csv :", C.STEAD_CSV)
df = pd.read_csv(C.STEAD_CSV, low_memory=False, usecols=["trace_name", "trace_category"])
print(df.shape, df.trace_category.value_counts().to_dict())
with h5py.File(C.STEAD_HDF5, "r") as f:
    print("traces in hdf5:", len(f["data"]))

/kaggle/working/repo
hdf5: /kaggle/input/datasets/prabhanshu22/dat-repo-data/merged.hdf5
csv : /kaggle/input/datasets/prabhanshu22/dat-repo-data/merged.csv
(1265657, 2) {'earthquake_local': 1030231, 'noise': 235426}
traces in hdf5: 235464


## The Author's Script Run

In [5]:
!python -u step2a_download_poha.py

/usr/local/lib/python3.13/dist-packages/obspy/clients/fdsn/client.py:257: ObsPyDeprecationWarning: IRIS is now EarthScope, please consider changing the FDSN client short URL to 'EARTHSCOPE'.
  warnings.warn(msg, ObsPyDeprecationWarning)
2021-06-01: 3 traces written
2021-06-02: 3 traces written
2021-06-03: 3 traces written
2021-06-04: 3 traces written
2021-06-05: 3 traces written
2021-06-06: 3 traces written
2021-06-07: 3 traces written
2021-06-08: 3 traces written
2021-06-09: 6 traces written
2021-06-10: 3 traces written
2021-06-11: 3 traces written
2021-06-12: 3 traces written
2021-06-13: 3 traces written
2021-06-14: 3 traces written
2021-06-15: 3 traces written
2021-06-16: 3 traces written
2021-06-17: 3 traces written
2021-06-18: 3 traces written
2021-06-19: 3 traces written
2021-06-20: 3 traces written
2021-06-21: 3 traces written
2021-06-22: 3 traces written
2021-06-23: 3 traces written
2021-06-24: 3 traces written
2021-06-25: 3 traces written
2021-06-26: 3 traces written
2021-06-2

In [6]:
!python -u step2b_prepare_STEAD_waveforms.py

total events in csv file: 1265657
total number of earthquakes: 153756
total number of noises: 235426
unique earthquakes drawn: 73544 / 100000
unique noise traces drawn: 81708 / 100000
============== 0 / 100000 ================= (0 s)
============== 1000 / 100000 ================= (125 s)
/kaggle/working/repo/step2b_prepare_STEAD_waveforms.py:75: RuntimeWarning: invalid value encountered in divide
  stacked_waveform = stacked_waveform / scaling_std
/kaggle/working/repo/step2b_prepare_STEAD_waveforms.py:76: RuntimeWarning: invalid value encountered in divide
  quake_waveform = quake_waveform / scaling_std
============== 2000 / 100000 ================= (246 s)
============== 3000 / 100000 ================= (365 s)
============== 4000 / 100000 ================= (483 s)
============== 5000 / 100000 ================= (608 s)
============== 6000 / 100000 ================= (731 s)
============== 7000 / 100000 ================= (849 s)
============== 8000 / 100000 ================= (964 s)
====

In [7]:
!python -u step2c_prepare_continous_STEAD_waveforms.py

components: ['BH1', 'BH2', 'BHZ']
POHA noise windows (60 s): 87191
total events in csv file: 1265657
total number of earthquakes: 153756
total number of noises: 235426
unique earthquakes drawn: 66651 / 87191
NOTE: dropping last 191 windows (same as the original code)
============== 0 / 87000 ================= (0 s)
============== 1000 / 87000 ================= (67 s)
============== 2000 / 87000 ================= (130 s)
============== 3000 / 87000 ================= (191 s)
============== 4000 / 87000 ================= (250 s)
============== 5000 / 87000 ================= (309 s)
============== 6000 / 87000 ================= (370 s)
============== 7000 / 87000 ================= (426 s)
============== 8000 / 87000 ================= (484 s)
============== 9000 / 87000 ================= (541 s)
============== 10000 / 87000 ================= (599 s)
============== 11000 / 87000 ================= (656 s)
============== 12000 / 87000 ================= (713 s)
============== 13000 / 87000 ====

In [8]:
!python -u step2d_combine_all_data.py

POHA mixtures: 87000 | STEAD mixtures: 100000 | total: 187000
unique samples after shuffle: 118217 / 187000 (63.2%)
saved: /kaggle/working/training_datasets/training_datasets_all_snr_40_unshuffled.hdf5


In [9]:
import h5py, numpy as np
with h5py.File(C.FINAL_DATASET, "r") as f:
    print({k: f[k].shape for k in f.keys()})
    print({k: v for k, v in f.attrs.items()})
    X = f["X_train"][:2000]; Y = f["Y_train"][:2000]
print("NaN in X:", int(np.isnan(X).sum()), "| mean std of X per component:", X.std(axis=1).mean().round(3))
!ls -lh /kaggle/working/training_datasets

{'X_train': (187000, 600, 3), 'Y_train': (187000, 600, 3), 'source_index': (187000,), 'time_new': (600,)}
{'n_poha': np.int64(87000), 'n_stead': np.int64(100000), 'n_unique': np.int64(118217), 'shuffle_seed': np.int64(0), 'with_replacement': np.True_}
NaN in X: 0 | mean std of X per component: 1.0
total 8.9G
-rw-r--r-- 1 root root 5.1G Nov  6 14:17 training_datasets_all_snr_40_unshuffled.hdf5
-rw-r--r-- 1 root root 1.9G Nov  6 14:16 training_datasets_STEAD_plus_POHA_snr_40_unshuffled.hdf5
-rw-r--r-- 1 root root 2.1G Nov  6 12:59 training_datasets_STEAD_waveform_snr_40.hdf5
